# Part B: reading synchrony backwards in real brain recordings

**How to run:** in Colab, choose Runtime > Run all. It takes about 30 minutes, mostly downloading. At the end, the last cell prints the numbers for your statement and saves figures for the README.

**The question:** can you tell from brain rhythms alone which parts are coupled, when nearby electrodes all hear the same sources?

**Data:** PhysioNet's EEG Motor Movement/Imagery Dataset: 109 volunteers, 64 electrodes, 160 samples a second. Run 1 is one minute resting with eyes open; run 2 is one minute with eyes closed. Please cite Schalk et al. (2004, *IEEE Transactions on Biomedical Engineering*) and Goldberger et al. (2000, *Circulation*).

Each step below starts with a short **What and why** note, so you can explain every result.

In [ ]:
!pip install -q mne mne-connectivity
import json, numpy as np, matplotlib.pyplot as plt, mne
from scipy.signal import hilbert
from scipy.stats import wilcoxon
from mne_connectivity import spectral_connectivity_epochs
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
mne.set_log_level("ERROR")
rng = np.random.default_rng(0)
REAR = ["PO3", "POz", "PO4", "O1", "Oz", "O2", "P1", "P2"]   # electrodes over the back of the head, where alpha is strongest
SFREQ = 160.0
print("ready")

## Step 1: load one volunteer

**What and why:** each volunteer has two one-minute files. `standardize` fixes the dataset's unusual electrode names (for example `O1..` becomes `O1`). A few volunteers were recorded at a different rate, so those are resampled to 160 Hz to make everyone comparable.

In [ ]:
def load_subject(s):
    files = mne.datasets.eegbci.load_data(s, [1, 2], update_path=False)
    raws = []
    for f in files:
        raw = mne.io.read_raw_edf(f, preload=True)
        mne.datasets.eegbci.standardize(raw)
        raw.set_montage("standard_1005")
        if raw.info["sfreq"] != SFREQ:
            raw.resample(SFREQ)
        raws.append(raw)
    return raws   # [eyes open, eyes closed]

eo, ec = load_subject(1)
print(len(eo.ch_names), "electrodes,", eo.times[-1].round(), "seconds each")

## Step 2: phases from the alpha rhythm

**What and why:** keep only the 8 to 12 Hz alpha band, the rhythm that strengthens when the eyes close. The Hilbert transform adds to each signal its quarter-period-shifted copy as an imaginary part. The signal becomes a point circling the origin: its angle is the phase and its distance from the origin is the amplitude.

**Sanity check:** alpha power should be higher with the eyes closed (Berger, 1929). If it isn't, something upstream is wrong.

In [ ]:
def alpha_phase(raw, picks=REAR):
    x = raw.copy().pick(picks).filter(8, 12).get_data()
    z = hilbert(x, axis=1)
    return np.unwrap(np.angle(z), axis=1), np.abs(z)

p_open = np.mean(alpha_phase(eo)[1] ** 2)
p_closed = np.mean(alpha_phase(ec)[1] ** 2)
print(f"alpha power, eyes closed / eyes open: {p_closed / p_open:.1f} times")

## Step 3: the old problem returns

**What and why:** in Part A, perfect synchrony erased the evidence of what caused it. Here it comes back in a new form. Neighbouring electrodes pick up the same brain sources at the same instant, so their phases move together whether or not anything is coupled. The phase-locking value counts any steady phase relation, including this zero-delay kind. So if it is high while imaginary coherency (which ignores zero delay) is low, most of the apparent synchrony is shared sources, not coupling.

In [ ]:
tri = np.tril_indices(len(REAR), -1)
def pair_measures(raw):
    ep = mne.make_fixed_length_epochs(raw, duration=2.0, preload=True).pick(REAR)
    plv, imc = spectral_connectivity_epochs(ep, method=["plv", "imcoh"], fmin=8, fmax=12,
                                            faverage=True, verbose=False)
    return (plv.get_data(output="dense")[:, :, 0][tri],
            np.abs(imc.get_data(output="dense")[:, :, 0][tri]))

plv1, imc1 = pair_measures(eo)
print(f"volunteer 1, eyes open, rear electrode pairs: phase-locking {plv1.mean():.2f}, |imaginary coherency| {imc1.mean():.2f}")

## Step 4: checking Nolte's claim on data where we know the answer

**What and why:** Nolte and colleagues (2004) showed that mixing with no time delay can only produce a *real* cross-spectrum. So the imaginary part of coherency ignores it, and only genuinely lagged interaction survives. Here two pairs of fake channels share one 10 Hz source: the first pair mixes it with no delay, the second with a delay. Imaginary coherency should be near 0 for the first pair and clearly above 0 for the second.

In [ ]:
t = np.arange(0, 60, 1 / SFREQ)
src = np.sin(2 * np.pi * 10 * t) + 0.5 * rng.normal(size=t.size)
noise = lambda: 0.3 * rng.normal(size=t.size)
def as_epochs(a, b):                      # (epochs, 2 channels, 2-second samples)
    return np.stack([a, b]).reshape(2, -1, 320).transpose(1, 0, 2)
for name, b in [("no delay", 0.7 * src + noise()), ("delayed", np.roll(src, 4) + noise())]:
    c = spectral_connectivity_epochs(as_epochs(src + noise(), b), method="imcoh", sfreq=SFREQ,
                                     fmin=8, fmax=12, faverage=True, verbose=False)
    print(f"{name}: |imaginary coherency| = {abs(c.get_data(output='dense')[1, 0, 0]):.3f}")

## Step 5: synchrony for all 109 volunteers (about 20 to 25 minutes)

**What and why:** each one-minute recording is cut into 2-second pieces. For every pair of rear electrodes (28 pairs), two measures are computed:
- **Phase-locking value:** this counts any consistent phase relation, including the fake, zero-delay kind.
- **|Imaginary coherency|:** this ignores zero-delay mixing.

Log alpha power is also recorded as a baseline.

In [ ]:
def features(raw):
    plv, imc = pair_measures(raw)
    return plv, imc, np.log(np.mean(alpha_phase(raw)[1] ** 2))

PLV, IMC, POW, LAB, GRP = [], [], [], [], []
for s in range(1, 110):
    try:
        for label, raw in enumerate(load_subject(s)):   # 0 = eyes open, 1 = eyes closed
            plv, imc, pw = features(raw)
            PLV.append(plv); IMC.append(imc); POW.append(pw); LAB.append(label); GRP.append(s)
    except Exception as e:
        print("skipped volunteer", s, "-", e)
PLV, IMC, POW, LAB, GRP = map(np.array, (PLV, IMC, POW, LAB, GRP))
np.savez("features.npz", PLV=PLV, IMC=IMC, POW=POW, LAB=LAB, GRP=GRP)
print(len(np.unique(GRP)), "volunteers loaded")

## Step 6: did synchrony change when the eyes closed?

**What and why:** for each volunteer, average the 28 pairs, giving one number for eyes open and one for eyes closed. The Wilcoxon signed-rank test asks whether the closed-minus-open differences lean one way more than chance would allow. It is a paired test, so each volunteer is compared only with themselves. Phase locking is shown alongside to see how differently the two measures behave.

In [ ]:
subs = [s for s in np.unique(GRP) if ((GRP == s) & (LAB == 0)).any() and ((GRP == s) & (LAB == 1)).any()]
def per_subject(M):
    o = np.array([M[(GRP == s) & (LAB == 0)].mean() for s in subs])
    c = np.array([M[(GRP == s) & (LAB == 1)].mean() for s in subs])
    return o, c
imc_o, imc_c = per_subject(IMC)
plv_o, plv_c = per_subject(PLV)
N = len(subs)
imc_up = int((imc_c > imc_o).sum()); imc_p = wilcoxon(imc_c, imc_o).pvalue
plv_up = int((plv_c > plv_o).sum()); plv_p = wilcoxon(plv_c, plv_o).pvalue
print(f"|imaginary coherency|: higher with eyes closed in {imc_up} of {N} volunteers, Wilcoxon p = {imc_p:.2g}")
print(f"phase-locking value:   higher with eyes closed in {plv_up} of {N} volunteers, Wilcoxon p = {plv_p:.2g}")
print(f"average level, eyes open: PLV {plv_o.mean():.2f}, |imcoh| {imc_o.mean():.2f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 4), sharey=False)
for a, (o, c, name) in zip(ax, [(plv_o, plv_c, "phase-locking value"), (imc_o, imc_c, "|imaginary coherency|")]):
    for i in range(N):
        a.plot([0, 1], [o[i], c[i]], color="grey", alpha=0.3)
    a.plot([0, 1], [np.median(o), np.median(c)], color="C0", lw=3)
    a.set_xticks([0, 1], ["eyes open", "eyes closed"]); a.set_title(name)
plt.tight_layout(); plt.savefig("fig_eyes_closed.png", dpi=150); plt.show()

## Step 7: can a classifier tell eyes open from closed in people it has never seen?

**What and why:** logistic regression learns a weighted sum of the 28 coherency values. GroupKFold splits by volunteer, so nobody's data is ever used both for training and for testing. This stops the model passing by memorising individuals. Scaling the features first matters, because the raw numbers differ hugely in size. The baseline uses alpha power alone: if it does just as well, the honest conclusion is that synchrony adds little beyond the rhythm's strength.

In [ ]:
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
cv = GroupKFold(n_splits=5)
acc_imc = cross_val_score(clf, IMC, LAB, groups=GRP, cv=cv)
acc_pow = cross_val_score(clf, POW.reshape(-1, 1), LAB, groups=GRP, cv=cv)
acc_both = cross_val_score(clf, np.column_stack([IMC, POW]), LAB, groups=GRP, cv=cv)
for name, a in [("imaginary coherency", acc_imc), ("alpha power alone", acc_pow), ("both", acc_both)]:
    print(f"{name:20s} {100 * a.mean():.0f}% (folds: {np.round(100 * a).astype(int)})")
plt.bar(["coherency", "power", "both"], [100 * acc_imc.mean(), 100 * acc_pow.mean(), 100 * acc_both.mean()])
plt.axhline(50, ls="--", color="grey"); plt.ylabel("accuracy on unseen volunteers (%)")
plt.savefig("fig_accuracy.png", dpi=150); plt.show()

## Your results for the statement

This cell prints the two slot values and checks whether the statement's wording still fits. Copy the printout into the chat with Claude.

In [ ]:
direction = "rose" if imc_c.mean() > imc_o.mean() else "fell"
count = imc_up if direction == "rose" else N - imc_up
finding = (f"rear-electrode coherency {direction} in {count} of {N} volunteers" if imc_p < 0.05
           else f"rear-electrode coherency did not change reliably (p = {imc_p:.2f})")
acc = round(100 * acc_imc.mean())
res = dict(plv_open=float(plv_o.mean()), imcoh_open=float(imc_o.mean()), N=N, imcoh_up=imc_up, imcoh_p=float(imc_p),
           plv_up=plv_up, plv_p=float(plv_p), acc_imcoh=float(acc_imc.mean()),
           acc_power=float(acc_pow.mean()), acc_both=float(acc_both.mean()), finding=finding)
json.dump(res, open("results.json", "w"), indent=1)
print("[FINDING] =", finding, f"({len(finding)} characters; keep it to 60 at most)")
print("[acc]     =", acc)
print()
print("Checks on the statement's wording:")
zero_lag = plv_o.mean() > 2 * imc_o.mean()
print(" - 'The old problem returned':", "supported" if zero_lag else "NOT clearly supported, so ask Claude to reword",
      f"(eyes open: phase-locking {plv_o.mean():.2f} against |imaginary coherency| {imc_o.mean():.2f})")
if acc_pow.mean() >= acc_imc.mean():
    print(f" - Alpha power alone scored {100 * acc_pow.mean():.0f}%, at least as well as coherency. Mention it honestly; ask Claude for the wording.")
else:
    print(f" - Coherency beat alpha power alone ({acc}% against {100 * acc_pow.mean():.0f}%).")

## Save your results

This bundles the results file and the two figures into one zip and downloads it, already arranged in `results/` and `figures/` folders: unzip it and drag both folders onto your repository's upload page.

In [ ]:
import zipfile
with zipfile.ZipFile("part_b_results.zip", "w") as z:
    z.write("results.json", "results/results.json")
    for f in ["fig_eyes_closed.png", "fig_accuracy.png"]:
        z.write(f, "figures/" + f)
try:
    from google.colab import files
    files.download("part_b_results.zip")
except ImportError:
    print("saved part_b_results.zip")